# Tutorial 4 · Models and data splits

AI in Rehabilitation and Healthcare (376-1229-00L) · HS 2026 · Lecture 5, Classification II: model selection and goodness metrics

The feature table from Tutorial 2 (`features_df_10s_3s.csv`: 24 people, 6 activities, 10 s windows with a 3 s step) becomes a classifier, scored four ways: on the rows it was trained on, on a held-out test set (split by rows, then by people), with a validation split for tuning, and with cross-validation by person, plain and nested. Read the scores and the confusion matrix each time, and ask what the number measures.

**Data.** `features_df_10s_3s.csv` is computed from the public MotionSense dataset (iPhone accelerometer and gyroscope at 50 Hz, 24 people, 6 activities) with Tutorial 2's recipe: a 10 Hz low-pass filter, then ten features per axis on 10-second windows with a 3-second step. Cite it as: Malekzadeh M, Clegg RG, Cavallaro A, Haddadi H. Mobile sensor data anonymization. *IoTDI '19*, ACM, 2019, pp. 49–58. doi:10.1145/3302505.3310068.

**With the course coach.** Run *T4.00* with your code from SCAI Learning (*Your data → Connect a notebook*). The coach then sees the code of each cell you run, and the *Report* cells send it numbers about your split, your scores and your model, never your data and never a cell's output. Ask it about a cell by its name, for example *"why does T4.05 split by person?"*. Without a code the notebook runs the same and sends nothing.


In [ ]:
# T4.00 Connect the course coach (optional)
# In SCAI Learning, open Your data → Connect a notebook and copy the six-letter code. Paste it
# between the quotes and run this cell. Left empty, nothing is installed and nothing is sent.
# Connected, the coach receives the code of each cell you run and the numbers the Report cells
# compute: never your data, never a cell's output.
COACH_CODE = ""
coach = globals().get("coach")          # connected already with the app's snippet? keep it
if COACH_CODE and coach is None:
    %pip install -q https://app.scai-learning.ch/companion.tar.gz
    import rehab_coach as coach
    coach.connect(COACH_CODE)
    coach.watch()
print("Coach connected: the cells you run from here on reach it." if coach else "Working without the coach.")


# **Load the tabular data**


In [ ]:
# T4.01 Mount Google Drive
# Colab only. On Jupyter on your own computer, skip this cell and set data_path below to your folder.
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
# T4.02 Load the feature table
# Load the featured dataset

import pandas as pd

data_path = '/content/drive/MyDrive/Colab Data/ai_in_rehab_2026_tutorial/'
load_path = data_path + 'features_df_10s_3s.csv'

features_df = pd.read_csv(load_path)
print(f"Loaded features_df from {load_path}")
print(features_df.shape)
features_df.head()


# **Train data (creating a classification model)**


In [ ]:
# T4.03 Train and score on the same rows
from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (accuracy_score, precision_score, recall_score, balanced_accuracy_score, f1_score, confusion_matrix)
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np


"""
Preparing data
"""

# Encode labels (activity) into numeric values
label_encoder = LabelEncoder()
y = label_encoder.fit_transform(features_df['activity'])

# Extract features and convert to numpy array
X = features_df.drop(['activity', 'subject', 'trial', 'window'], axis=1).to_numpy()

# Store subject info
subjects = features_df['subject'].to_numpy()

"""
Training the model
"""

# Creating an object of the model with set hyper-parameters
model = RandomForestClassifier(
    n_estimators=10,      # number of trees
    max_depth=None,       # or set e.g. max_depth=10
    random_state=42
)

# Training the model on the data
model.fit(X, y)

"""
Evaluation of the model
"""

# Predict labels
y_pred = model.predict(X)

# Compute metrics
accuracy = accuracy_score(y, y_pred)
precision = precision_score(y, y_pred, average='macro')
recall = recall_score(y, y_pred, average='macro')
balanced_accuracy = balanced_accuracy_score(y, y_pred)
f1 = f1_score(y, y_pred, average='macro')

# Print metrics
print(f'Accuracy: {100*accuracy:.2f}%')
print(f'Precision: {100*precision:.2f}%')
print(f'Recall: {100*recall:.2f}%')
print(f'Balanced Accuracy: {100*balanced_accuracy:.2f}%')
print(f'F1 Score: {100*f1:.2f}%')

"""
Create confusion matrix
"""

conf_matrix = confusion_matrix(y, y_pred)

# Recover original class labels from the encoder
class_labels = label_encoder.classes_

# Create a small confusion matrix figure
plt.figure(figsize=(6, 4))  # adjust size as needed (width, height)
sns.heatmap(conf_matrix,
            annot=True,         # show numbers
            fmt='d',            # integer format
            cmap='Blues',       # color map
            xticklabels=class_labels,
            yticklabels=class_labels)

plt.xlabel("Predicted Label")
plt.ylabel("True Label")
plt.title("Confusion Matrix")
plt.tight_layout()  # ensures labels fit well
plt.show()


In [ ]:
# T4.04 Report the training score
# Numbers about the cells above, computed here and sent to the course coach (nothing else is
# sent, and nothing at all without T4.00).
if coach:
    idx = np.arange(len(y))                                   # scored on the rows it was trained on
    print(f"Zero-rate baseline (always the largest class): {100 * coach.baseline(y):.2f}%")
    coach.report((idx, idx), subjects, y, model=model, tutorial="T4",
                 scores={"accuracy": accuracy, "balanced_accuracy": balanced_accuracy, "f1_macro": f1},
                 lesson={"training_score": accuracy})
else:
    print("Not connected: nothing sent. Run T4.00 with your code to use the coach.")


## Question) Why so perfect? Do we actually have a perfect classifier?


# **Test data**


In [ ]:
# T4.05 Split: by rows, or by people
from sklearn.model_selection import train_test_split
from sklearn.model_selection import GroupShuffleSplit


# [Option 1]
# Split the dataset into fixed training and test sets (by rows: a person's windows can land on both sides)
train_idx, test_idx = train_test_split(np.arange(len(y)), test_size=0.3, random_state=42)

# # [Option 2]
# # Split the dataset into fixed training and test sets (based on subjects)
# splitter = GroupShuffleSplit(n_splits=1, test_size=0.3, random_state=42)
# train_idx, test_idx = next(splitter.split(X, y, groups=subjects))

X_train, X_test = X[train_idx], X[test_idx]
y_train, y_test = y[train_idx], y[test_idx]
print('People on both sides of the split:', len(set(subjects[train_idx]) & set(subjects[test_idx])))


"""
Training the model
"""

# Creating an object of the model with set hyper-parameters
model = RandomForestClassifier(
    n_estimators=10,       # number of trees
    max_depth=None,        # or set e.g. max_depth=10
    n_jobs=-1,
    random_state=42
)

# Training the model on the train data
model.fit(X_train, y_train)


"""
Testing the model on test data
"""

# Predict test labels
y_pred = model.predict(X_test)

# Compute metrics
accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred, average='macro')
recall = recall_score(y_test, y_pred, average='macro')
balanced_accuracy = balanced_accuracy_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred, average='macro')

# Print metrics
print(f'Accuracy: {100*accuracy:.2f}%')
print(f'Precision: {100*precision:.2f}%')
print(f'Recall: {100*recall:.2f}%')
print(f'Balanced Accuracy: {100*balanced_accuracy:.2f}%')
print(f'F1 Score: {100*f1:.2f}%')


"""
Create confusion matrix
"""

conf_matrix = confusion_matrix(y_test, y_pred)
class_labels = label_encoder.classes_   # original string labels

# Create a small confusion matrix figure
plt.figure(figsize=(6, 4))  # adjust size as needed (width, height)
sns.heatmap(conf_matrix,
            annot=True,         # show numbers
            fmt='d',            # integer format
            cmap='Blues',       # color map
            xticklabels=class_labels,
            yticklabels=class_labels)

plt.xlabel("Predicted Label")
plt.ylabel("True Label")
plt.title("Confusion Matrix")
plt.tight_layout()  # ensures labels fit well
plt.show()


In [ ]:
# T4.06 Report the split and its scores
# Numbers about the cells above, computed here and sent to the course coach (nothing else is
# sent, and nothing at all without T4.00).
if coach:
    coach.report((train_idx, test_idx), subjects, y, model=model, tutorial="T4",
                 scores={"accuracy": accuracy, "balanced_accuracy": balanced_accuracy, "f1_macro": f1})
else:
    print("Not connected: nothing sent. Run T4.00 with your code to use the coach.")


### Exercise) Try splitting data by labels, and subjects. How are they different?
### Exercise) Try different percentages of test data and see how it affects your performance.


# **Validation data (hyper-parameter tuning)**
### Exercise) Complement the lists of 'n_estimators', and 'max_depth' for hyper-par tuning.


In [ ]:
# T4.07 Validation split for tuning
# Split the dataset into fixed training+validation and test sets (based on subjects)
splitter = GroupShuffleSplit(n_splits=1, test_size=0.3, random_state=42)
train_idx, test_idx = next(splitter.split(X, y, groups=subjects))
X_train_full, X_test = X[train_idx], X[test_idx]
y_train_full, y_test = y[train_idx], y[test_idx]
subjects_train_full = subjects[train_idx]  # subjects for the training subset

# Split the training+validation data into training and validation sets (also by subjects)
splitter_val = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx2, val_idx = next(splitter_val.split(X_train_full, y_train_full, groups=subjects_train_full))
X_train, X_val = X_train_full[train_idx2], X_train_full[val_idx]
y_train, y_val = y_train_full[train_idx2], y_train_full[val_idx]
subjects_train, subjects_val = subjects_train_full[train_idx2], subjects_train_full[val_idx]


"""
Hyperparameter tuning on validation data (based on F1-macro)
"""

best_score = 0
best_params = {}

for n_estimators in [10, 50]:        # <------------------------------------------------------------------- Complement
    for max_depth in [None, 5, 10]:       # <---------------------------------------------------------------Complement
        model = RandomForestClassifier(
            n_estimators=n_estimators,
            max_depth=max_depth,
            min_samples_split=2,
            min_samples_leaf=1,
            random_state=42,
            n_jobs=-1
        )
        model.fit(X_train, y_train)
        y_val_pred = model.predict(X_val)
        val_f1 = f1_score(y_val, y_val_pred, average='macro')
        print(f"n_estimators: {n_estimators}, max_depth: {max_depth} — F1-macro: {100*val_f1:.2f}%")
        if val_f1 > best_score:
            best_score = val_f1
            best_params = {'n_estimators': n_estimators, 'max_depth': max_depth}

print("\n\n\nBest hyperparameters found on validation set (based on F1-macro):")
print(best_params)
print(f"Validation F1-macro: {100*best_score:.2f}%\n")

"""
Training the model on the combined training + validation data
"""

model = RandomForestClassifier(
    n_estimators=best_params['n_estimators'],
    max_depth=best_params['max_depth'],
    min_samples_split=2,
    min_samples_leaf=1,
    random_state=42,
    n_jobs=-1
)

# Combine training and validation sets for final model
X_train_final = np.concatenate([X_train, X_val])
y_train_final = np.concatenate([y_train, y_val])

# Training the model on the train data
model.fit(X_train_final, y_train_final)

"""
Testing the model on test data
"""

y_pred = model.predict(X_test)

# Compute metrics
accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred, average='macro')
recall = recall_score(y_test, y_pred, average='macro')
balanced_accuracy = balanced_accuracy_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred, average='macro')

# Print metrics
print(f'Accuracy: {100*accuracy:.2f}%')
print(f'Precision: {100*precision:.2f}%')
print(f'Recall: {100*recall:.2f}%')
print(f'Balanced Accuracy: {100*balanced_accuracy:.2f}%')
print(f'F1 Score: {100*f1:.2f}%')

"""
Create confusion matrix
"""

conf_matrix = confusion_matrix(y_test, y_pred)
class_labels = label_encoder.classes_   # original string labels
# Create a small confusion matrix figure
plt.figure(figsize=(6, 4))  # adjust size as needed (width, height)
sns.heatmap(conf_matrix,
            annot=True,         # show numbers
            fmt='d',            # integer format
            cmap='Blues',       # color map
            xticklabels=class_labels,
            yticklabels=class_labels)

plt.xlabel("Predicted Label")
plt.ylabel("True Label")
plt.title("Confusion Matrix")
plt.tight_layout()  # ensures labels fit well
plt.show()


In [ ]:
# T4.08 Report the validation split
# Numbers about the cells above, computed here and sent to the course coach (nothing else is
# sent, and nothing at all without T4.00).
if coach:
    # the validation split inside the training people; the test people stay locked, so no test score
    coach.report((train_idx2, val_idx), subjects_train_full, y_train_full, tutorial="T4")
else:
    print("Not connected: nothing sent. Run T4.00 with your code to use the coach.")


# **Cross validation (train and test)**


In [ ]:
# T4.09 Cross-validation by person
from sklearn.model_selection import GroupKFold


"""
Cross-validation (subject-based, 5 folds)
"""

# Set up group-based cross-validation (split by subjects)
cv = GroupKFold(n_splits=5)

# Arrays to store results for each fold
accuracy_scores = []
precision_scores = []
recall_scores = []
balanced_accuracy_scores = []
f1_scores = []

# Arrays to store predictions for each fold
y_true_all = []
y_pred_all = []

# Loop through the cross-validation splits
for fold, (train_index, test_index) in enumerate(cv.split(X, y, groups=subjects), start=1):
    X_train, X_test = X[train_index], X[test_index]
    y_train, y_test = y[train_index], y[test_index]

    # Creating an object of the model with set hyper-parameters
    model = RandomForestClassifier(
        n_estimators=150,       # number of trees
        max_depth=10,        # or set e.g. max_depth=10
        n_jobs=-1,
        random_state=42
    )

    # Training the model using the train data in the current fold
    model.fit(X_train, y_train)

    # Evaluate the model using the test data in the current fold
    y_pred = model.predict(X_test)

    y_true_all.extend(y_test)  # Store true labels for the current fold
    y_pred_all.extend(y_pred)  # Store predicted labels for the current fold

    # Calculate classification metrics for this split
    accuracy_scores.append(accuracy_score(y_test, y_pred))
    precision_scores.append(precision_score(y_test, y_pred, average='macro'))
    recall_scores.append(recall_score(y_test, y_pred, average='macro'))
    balanced_accuracy_scores.append(balanced_accuracy_score(y_test, y_pred))
    f1_scores.append(f1_score(y_test, y_pred, average='macro'))

    print(
        f"Fold {fold} — "
        f"Accuracy: {100*accuracy_scores[-1]:.2f}%, "
        f"Precision: {100*precision_scores[-1]:.2f}%, "
        f"Recall: {100*recall_scores[-1]:.2f}%, "
        f"Balanced Accuracy: {100*balanced_accuracy_scores[-1]:.2f}%, "
        f"F1-macro: {100*f1_scores[-1]:.2f}%"
    )

print(
    f"Overall: "
    f"\n\tAccuracy: {100*np.mean(accuracy_scores):.2f}% ± {100*np.std(accuracy_scores):.2f}%, "
    f"\n\tPrecision: {100*np.mean(precision_scores):.2f}% ± {100*np.std(precision_scores):.2f}%, "
    f"\n\tRecall: {100*np.mean(recall_scores):.2f}% ± {100*np.std(recall_scores):.2f}%, "
    f"\n\tBalanced Accuracy: {100*np.mean(balanced_accuracy_scores):.2f}% ± {100*np.std(balanced_accuracy_scores):.2f}%, "
    f"\n\tF1-macro: {100*np.mean(f1_scores):.2f}% ± {100*np.std(f1_scores):.2f}%"
)


"""
Create confusion matrix for all folds aggregated
"""

conf_matrix = confusion_matrix(y_true_all, y_pred_all)
class_labels = label_encoder.classes_   # original string labels

# Create a small confusion matrix figure
plt.figure(figsize=(6, 4))  # adjust size as needed (width, height)
sns.heatmap(conf_matrix,
            annot=True,         # show numbers
            fmt='d',            # integer format
            cmap='Blues',       # color map
            xticklabels=class_labels,
            yticklabels=class_labels)

plt.xlabel("Predicted Label")
plt.ylabel("True Label")
plt.title("Confusion Matrix")
plt.tight_layout()  # ensures labels fit well
plt.show()


In [ ]:
# T4.10 Report the cross-validation
# Numbers about the cells above, computed here and sent to the course coach (nothing else is
# sent, and nothing at all without T4.00).
if coach:
    coach.report(cv, subjects, y, tutorial="T4",
                 scores={"balanced_accuracy": float(np.mean(balanced_accuracy_scores)), "f1_macro": float(np.mean(f1_scores))})
else:
    print("Not connected: nothing sent. Run T4.00 with your code to use the coach.")


# **Nested cross validation (test and validation in loops)**


In [ ]:
# T4.11 Nested cross-validation
from sklearn.model_selection import GridSearchCV


"""
Nested Cross-Validation (subject-based)
Outer loop: test set
Inner loop: hyperparameter tuning via GridSearchCV
"""

# Set up outer group-based cross-validation (split by subjects)
outer_cv = GroupKFold(n_splits=5)

# Arrays to store results for each outer fold
accuracy_scores = []
precision_scores = []
recall_scores = []
balanced_accuracy_scores = []
f1_scores = []

# Arrays to store predictions across outer folds
y_true_all = []
y_pred_all = []

# Define hyperparameter grid for RandomForest
param_grid = {
    'n_estimators': [10, 50],
    'max_depth': [None, 5, 10]
}

# Outer CV loop
for outer_fold, (train_outer_idx, test_outer_idx) in enumerate(outer_cv.split(X, y, groups=subjects), start=1):
    X_train_outer, X_test_outer = X[train_outer_idx], X[test_outer_idx]
    y_train_outer, y_test_outer = y[train_outer_idx], y[test_outer_idx]
    subjects_train_outer = subjects[train_outer_idx]

    # Inner CV (for hyperparameter tuning) — also group-based
    inner_cv = GroupKFold(n_splits=3)

    # Define base model
    model = RandomForestClassifier(
        n_jobs=-1,
        random_state=42
    )

    # Grid search over the inner folds
    grid_search = GridSearchCV(
        estimator=model,
        param_grid=param_grid,
        scoring='f1_macro',
        cv=list(inner_cv.split(X_train_outer, y_train_outer, groups=subjects_train_outer)),   # a list: newer scikit-learn cannot send a generator to parallel workers
        n_jobs=-1
    )
    grid_search.fit(X_train_outer, y_train_outer)

    best_model = grid_search.best_estimator_
    best_params = grid_search.best_params_
    best_score = grid_search.best_score_

    print(f"Outer Fold {outer_fold} — Best Params: {best_params}, Inner Mean F1: {100*best_score:.2f}%")

    # Test best model on outer test data
    y_test_pred = best_model.predict(X_test_outer)

    y_true_all.extend(y_test_outer)
    y_pred_all.extend(y_test_pred)

    # Calculate metrics for this outer fold
    accuracy_scores.append(accuracy_score(y_test_outer, y_test_pred))
    precision_scores.append(precision_score(y_test_outer, y_test_pred, average='macro'))
    recall_scores.append(recall_score(y_test_outer, y_test_pred, average='macro'))
    balanced_accuracy_scores.append(balanced_accuracy_score(y_test_outer, y_test_pred))
    f1_scores.append(f1_score(y_test_outer, y_test_pred, average='macro'))

    print(
        f"Outer Fold {outer_fold} — "
        f"Accuracy: {100*accuracy_scores[-1]:.2f}%, "
        f"Precision: {100*precision_scores[-1]:.2f}%, "
        f"Recall: {100*recall_scores[-1]:.2f}%, "
        f"Balanced Accuracy: {100*balanced_accuracy_scores[-1]:.2f}%, "
        f"F1-macro: {100*f1_scores[-1]:.2f}%"
    )

print(
    f"Overall: "
    f"\n\tAccuracy: {100*np.mean(accuracy_scores):.2f}% ± {100*np.std(accuracy_scores):.2f}%, "
    f"\n\tPrecision: {100*np.mean(precision_scores):.2f}% ± {100*np.std(precision_scores):.2f}%, "
    f"\n\tRecall: {100*np.mean(recall_scores):.2f}% ± {100*np.std(recall_scores):.2f}%, "
    f"\n\tBalanced Accuracy: {100*np.mean(balanced_accuracy_scores):.2f}% ± {100*np.std(balanced_accuracy_scores):.2f}%, "
    f"\n\tF1-macro: {100*np.mean(f1_scores):.2f}% ± {100*np.std(f1_scores):.2f}%"
)


"""
Create confusion matrix for all outer folds aggregated
"""

conf_matrix = confusion_matrix(y_true_all, y_pred_all)
class_labels = label_encoder.classes_

plt.figure(figsize=(6, 4))
sns.heatmap(conf_matrix,
            annot=True,
            fmt='d',
            cmap='Blues',
            xticklabels=class_labels,
            yticklabels=class_labels)

plt.xlabel("Predicted Label")
plt.ylabel("True Label")
plt.title("Aggregated Confusion Matrix (Nested CV with Grid Search)")
plt.tight_layout()
plt.show()


In [ ]:
# T4.12 Report the nested cross-validation
# Numbers about the cells above, computed here and sent to the course coach (nothing else is
# sent, and nothing at all without T4.00).
if coach:
    coach.report(outer_cv, subjects, y, tutorial="T4", scores={"f1_macro": float(np.mean(f1_scores))})
else:
    print("Not connected: nothing sent. Run T4.00 with your code to use the coach.")


# Exercise!

Let's experiment on the last cell!
1) Add more options to the 'param_grid' for the hyper parameters. You can start with this:


```python
param_grid = {
    'n_estimators': [50, 100, 200],       # number of trees
    'max_depth': [None, 10, 20, 30],      # depth of each tree
    'min_samples_split': [2, 5, 10],      # minimum samples to split an internal node
    'min_samples_leaf': [1, 2, 4],        # minimum samples required at a leaf node
    'max_features': ['sqrt', 'log2'],     # number of features considered at each split
    'bootstrap': [True, False]            # whether bootstrap samples are used
}
```

Doing so you will notice how long it takes to do a nested cross validation with many hyper-parameters!

So, also replace 'GridSearchCV' with 'RandomizedSearchCV'. Here is an example:
```python
from sklearn.model_selection import RandomizedSearchCV

# Randomized search over the inner folds
random_search = RandomizedSearchCV(
    estimator=model,
    param_distributions=param_grid,   # same grid as before
    n_iter=20,                        # number of random combinations to try
    scoring='f1_macro',
    cv=inner_cv.split(X_train_outer, y_train_outer, groups=subjects_train_outer),
    n_jobs=-1,
    random_state=42
)
```

2) Make a similar nested CV pipeline including subject splits, and hyper-parameter tuning, with a classifier of your own choise (not Random Forest).
